# Cohort Trial Run 3 Inference Monitor

Dashboard/control notebook for cohort slide processing. Execution, status, ETA, preview, and merge logic live in `scripts/run_cohort_trial3_inference.py`.

In [ ]:
from pathlib import Path
import json
import os
import signal
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Image as IPyImage, Markdown, clear_output, display

# Resolve repo whether Jupyter starts in the repo root or notebooks/.
REPO = Path.cwd().resolve()
if not (REPO / "scripts" / "run_cohort_trial3_inference.py").exists() and (REPO.parent / "scripts" / "run_cohort_trial3_inference.py").exists():
    REPO = REPO.parent
if not (REPO / "scripts" / "run_cohort_trial3_inference.py").exists():
    REPO = Path.home() / "Desktop" / "inference_yolo_sam_cellseg"

CONFIG = REPO / "configs" / "cohort_inference_slides.yaml"
SCRIPT = REPO / "scripts" / "run_cohort_trial3_inference.py"
sys.path.insert(0, str(REPO / "scripts"))
import run_cohort_trial3_inference as cohort

config = cohort.load_config(CONFIG)
settings = cohort.settings_from_config(config)
OUTPUT_ROOT = settings["output_root"]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("REPO:", REPO)
print("CONFIG:", CONFIG)
print("SCRIPT:", SCRIPT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("workers:", settings["workers"], "gpu_ids:", settings["gpu_ids"])
print("refresh_seconds:", settings["refresh_seconds"])

In [ ]:
# Dry-run table: slides to run/resume, skipped slides, and tile counts.
records = cohort.dry_run_records(config)
dry_run_df = pd.DataFrame(records)
cols = ["slide_name", "action", "reason", "total_tiles", "completed_tiles", "workers", "tile_dir", "output_dir"]

display(Markdown("## Dry Run"))
display(dry_run_df[cols])

display(Markdown("### Slides to run or resume"))
display(dry_run_df[dry_run_df["action"].isin(["run", "resume"])][cols])

display(Markdown("### Slides skipped or unavailable"))
display(dry_run_df[~dry_run_df["action"].isin(["run", "resume"])][cols])

In [ ]:
# Launch cohort inference through the reusable script.
# Re-running this cell is safe: the script lock prevents duplicate active cohort runs.
launcher_log = OUTPUT_ROOT / "cohort_launcher.log"
lock_path = OUTPUT_ROOT / "_cohort_RUN.lock"


def pid_alive(pid):
    try:
        os.kill(int(pid), 0)
        return True
    except Exception:
        return False

active_pid = None
if lock_path.exists():
    try:
        active_pid = json.loads(lock_path.read_text()).get("pid")
    except Exception:
        active_pid = None

if active_pid and pid_alive(active_pid):
    print(f"Cohort runner already active pid={active_pid}")
    COHORT_PROC = None
else:
    log_handle = launcher_log.open("a", buffering=1)
    cmd = [sys.executable, str(SCRIPT), "--config", str(CONFIG), "--run"]
    COHORT_PROC = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    print("Launched cohort runner pid=", COHORT_PROC.pid)
    print("Log:", launcher_log)

In [ ]:
# Live dashboard. Stop this cell manually when you are done watching.
REFRESH_SECONDS = settings["refresh_seconds"]
LOG_TAIL_LINES = settings["log_tail_lines"]


def read_tail(path, lines=40):
    path = Path(path)
    if not path.exists():
        return ""
    try:
        return "\n".join(path.read_text(errors="replace").splitlines()[-lines:])
    except Exception:
        return ""


def format_seconds(seconds):
    try:
        seconds = float(seconds)
    except Exception:
        return "n/a"
    if pd.isna(seconds):
        return "n/a"
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


def eta_summary(df, seconds_col, human_col):
    if df.empty:
        return "calculating"
    values = pd.to_numeric(df.get(seconds_col, pd.Series(dtype=float)), errors="coerce").dropna()
    if not values.empty:
        return format_seconds(values.max())
    labels = [str(x) for x in df.get(human_col, pd.Series(dtype=str)).dropna() if str(x).strip()]
    if any(label == "warming up" for label in labels):
        return "warming up"
    if any(label == "calculating" for label in labels):
        return "calculating"
    return labels[-1] if labels else "calculating"


def timing_summary(slide_name, worker_id):
    timing_path = OUTPUT_ROOT / slide_name / "_status" / f"worker_{int(worker_id):02d}_timing.csv"
    if not timing_path.exists() or timing_path.stat().st_size == 0:
        return pd.Series({"failed_tiles": 0, "timed_cells": 0})
    try:
        df = pd.read_csv(timing_path)
    except Exception:
        return pd.Series({"failed_tiles": 0, "timed_cells": 0})
    if df.empty:
        return pd.Series({"failed_tiles": 0, "timed_cells": 0})
    success = df["success"].astype(str).str.lower().eq("true") if "success" in df else pd.Series([], dtype=bool)
    failed = int((~success).sum()) if len(success) else 0
    cells = pd.to_numeric(df.loc[success, "num_instances"], errors="coerce").fillna(0).sum() if "num_instances" in df else 0
    return pd.Series({"failed_tiles": failed, "timed_cells": int(cells)})


def prepare_workers(snapshot):
    workers = pd.DataFrame(snapshot["workers"])
    if workers.empty:
        return workers
    timing = workers.apply(lambda row: timing_summary(row["slide_name"], row["worker_id"]), axis=1)
    workers = pd.concat([workers, timing], axis=1)
    workers["elapsed_seconds"] = pd.to_numeric(workers.get("elapsed_seconds", 0), errors="coerce").fillna(0)
    workers["elapsed"] = workers["elapsed_seconds"].apply(format_seconds)
    workers["eta_median"] = workers.get("eta_human_median", pd.Series("calculating", index=workers.index)).fillna("calculating")
    workers["eta_conservative"] = workers.get("eta_human_conservative", pd.Series("calculating", index=workers.index)).fillna("calculating")
    workers["cells_per_sec"] = workers.apply(
        lambda row: (row["timed_cells"] / row["elapsed_seconds"]) if row.get("elapsed_seconds", 0) else 0,
        axis=1,
    )
    return workers


def add_slide_timing(slides, workers):
    if slides.empty or workers.empty:
        return slides
    rows = []
    for slide_name, group in workers.groupby("slide_name", dropna=False):
        rows.append({
            "slide_name": slide_name,
            "elapsed": format_seconds(pd.to_numeric(group["elapsed_seconds"], errors="coerce").max()),
            "eta_median": eta_summary(group, "eta_seconds_median", "eta_human_median"),
            "eta_conservative": eta_summary(group, "eta_seconds_conservative", "eta_human_conservative"),
            "failed_tiles": int(pd.to_numeric(group.get("failed_tiles", 0), errors="coerce").fillna(0).sum()),
        })
    return slides.merge(pd.DataFrame(rows), on="slide_name", how="left")


def dashboard_once():
    snapshot = cohort.monitor_snapshot(config)
    workers = prepare_workers(snapshot)
    slides = add_slide_timing(pd.DataFrame(snapshot["slides"]), workers)
    clear_output(wait=True)

    c = snapshot["cohort"]
    active_workers = workers[workers["status"].isin(["running", "restarting", "finishing"])] if not workers.empty else workers
    timing_source = active_workers if not active_workers.empty else workers
    cohort_elapsed = format_seconds(pd.to_numeric(timing_source.get("elapsed_seconds", pd.Series(dtype=float)), errors="coerce").max()) if not timing_source.empty else "n/a"
    cohort_eta_median = eta_summary(timing_source, "eta_seconds_median", "eta_human_median")
    cohort_eta_conservative = eta_summary(timing_source, "eta_seconds_conservative", "eta_human_conservative")

    display(Markdown(f"""
## Cohort Status

**Tiles:** `{c['tiles_done']}/{c['tiles_total']}` (`{c['percent_complete']}%`)  
**Slides:** `{c['slides_completed']}` completed | `{c['slides_running']}` running | `{c['slides_failed']}` failed  
**Elapsed:** `{cohort_elapsed}` | **ETA median:** `{cohort_eta_median}` | **ETA conservative:** `{cohort_eta_conservative}`  
**Updated:** `{c['updated_at']}`  
**Output:** `{c['output_root']}`
"""))

    if not slides.empty:
        slide_cols = [col for col in [
            "slide_name", "status", "tiles_done", "tiles_total", "percent_complete",
            "elapsed", "eta_median", "eta_conservative", "failed_tiles", "updated_at", "error"
        ] if col in slides]
        display(Markdown("### Slide Status"))
        display(slides[slide_cols])

    if not workers.empty:
        worker_cols = [col for col in [
            "slide_name", "worker_id", "status", "tiles_done", "tiles_total", "percent_complete",
            "elapsed", "eta_median", "eta_conservative", "tiles_per_minute", "cells_per_sec",
            "failed_tiles", "current_tile", "error"
        ] if col in workers]
        display(Markdown("### Worker Status"))
        display(workers[worker_cols])

        preview_rows = workers[["slide_name", "worker_id", "latest_preview_image_path"]].dropna()
        preview_rows = preview_rows[preview_rows["latest_preview_image_path"].astype(str) != ""]
        if not preview_rows.empty:
            display(Markdown("### Latest Preview Per Worker"))
            for _, row in preview_rows.tail(12).iterrows():
                path = Path(row["latest_preview_image_path"])
                if path.exists():
                    display(Markdown(f"**{row['slide_name']} worker {int(row['worker_id']):02d}**"))
                    display(IPyImage(filename=str(path), width=360))

    launcher_tail = read_tail(OUTPUT_ROOT / "cohort_launcher.log", LOG_TAIL_LINES)
    if launcher_tail:
        display(Markdown("### Recent Launcher Log"))
        print(launcher_tail)

    if not workers.empty:
        running = workers[workers["status"].isin(["running", "restarting", "finishing"])].tail(1)
        if not running.empty:
            row = running.iloc[0]
            worker_log = OUTPUT_ROOT / row["slide_name"] / "logs" / f"worker_{int(row['worker_id']):02d}.log"
            tail = read_tail(worker_log, LOG_TAIL_LINES)
            if tail:
                display(Markdown("### Recent Worker Log"))
                print(tail)


while True:
    dashboard_once()
    time.sleep(REFRESH_SECONDS)


In [ ]:
# One-shot status snapshot, useful after restarting the notebook kernel.
snapshot = cohort.monitor_snapshot(config)
print(json.dumps(snapshot["cohort"], indent=2))
if snapshot["slides"]:
    display(pd.DataFrame(snapshot["slides"]))
if snapshot["workers"]:
    display(pd.DataFrame(snapshot["workers"]))